# PCMCI · LPCMCI: 레짐 없는 인과 그래프와 답지 채점

`RCBNB-MB.ipynb` 와 같은 조건에서 tigramite 의 두 방법을 2015-01-04 ~ 최신 전체에 한 번씩 적용한다.

- 데이터·변환·답지: `analysis/causal_common.py` (RCBNB-MB 와 같은 함수). τ_max = 4, 유의수준 0.05, ParCorr(선형 편상관).
- **PCMCI**: 시차 링크만 찾는다(tau_min = 1). 모든 링크가 과거 → 현재라 방향이 정해진다. 같은 주의 원인은 원천적으로 잡지 못한다.
- **LPCMCI**: 같은 주(lag 0) 링크와 **관측되지 않은 공통 원인(잠재 교란)** 을 허용한다. 결과는 PAG 라 끝점 표시가 세 가지다:
  `-` 꼬리(이쪽이 원인), `>` 화살촉(이쪽은 원인이 아님), `o` 미정. 예: `A o-> B` 는 "B 는 A 의 원인이 아니다, A 가 B 의 원인인지는 모른다".
- 채점: `docs/commodity_weekly_drivers.xlsx` 의 주별 원인표(2023-01 ~ 2026-10). 무작위 변수 집합의 기대 적중률(우연 기준선)을 함께 적는다.

이미 본 기간을 탐색하는 분석이다. 예측 성능 평가가 아니다.

In [ ]:
import hashlib, os, pickle, time, warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from matplotlib import font_manager
from scipy.special import comb
from tigramite import data_processing as pp
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.lpcmci import LPCMCI
from tigramite.pcmci import PCMCI

from causal_common import ALPHA, MISSING, ROOT, SERIES, TAU, load_key, make_panel  # RCBNB-MB 와 같은 데이터·변환·답지

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"  # 병렬 작업 프로세스에도 적용
CACHE = ROOT / "data/07_reporting/pcmci_lpcmci"  # 그래프 캐시(데이터·방법·설정의 해시). discover 를 바꾸면 이 폴더를 지운다.
CACHE.mkdir(parents=True, exist_ok=True)

METHODS = ["PCMCI", "LPCMCI"]
LPCMCI_OPTS = {"max_p_global": 3}  # 조건 집합 크기 ≤ 3. 기본 설정(제한 없음)은 원자재 하나에 30분을 넘겨 끝나지 않았다

panels = {s: make_panel(s)[0] for s in SERIES}
keys = {s: load_key(s) for s in SERIES}
for s, X in panels.items():
    print(f"{s}: T={len(X)}, 변수 {X.shape[1]}개 (가격 포함), 결측 비율={X.isna().mean().mean():.3f}")

## 1. 인과 그래프

`graph[i, j, τ]` 는 X^i(t−τ) 와 X^j(t) 사이 링크의 끝점 표시(왼쪽 = i 쪽, 오른쪽 = j 쪽). 결측 주는 `missing_flag` 로 검정에서 뺀다.

In [ ]:
def discover(method, X, opts):
    path = CACHE / (hashlib.md5(X.tobytes() + repr((method, sorted(opts.items()), TAU, ALPHA)).encode()).hexdigest() + ".pkl")
    if path.exists():
        return pickle.loads(path.read_bytes())
    frame = pp.DataFrame(np.nan_to_num(X, nan=MISSING), missing_flag=MISSING)
    test = ParCorr(significance="analytic")
    t0 = time.time()
    if method == "PCMCI":
        out = PCMCI(frame, test, verbosity=0).run_pcmci(tau_min=1, tau_max=TAU, pc_alpha=ALPHA, alpha_level=ALPHA)
    else:
        out = LPCMCI(frame, test, verbosity=0).run_lpcmci(tau_min=0, tau_max=TAU, pc_alpha=ALPHA, **opts)
    res = out["graph"], out["val_matrix"], time.time() - t0
    path.write_bytes(pickle.dumps(res))
    return res


jobs = [(s, m) for s in SERIES for m in METHODS]
results = dict(zip(jobs, Parallel(n_jobs=len(jobs))(
    delayed(discover)(m, panels[s].values, LPCMCI_OPTS if m == "LPCMCI" else {}) for s, m in jobs)))
for (s, m), (g, _, sec) in results.items():
    marks = pd.Series(g[g != ""]).value_counts().to_dict()
    print(f"{s:7s} {m:6s} 계산 {sec:7.1f}초  링크 끝점 표시별 개수 {marks}")
    if m == "PCMCI":
        assert (g[:, :, 0] == "").all() and set(marks) <= {"-->"}  # tau_min=1: 같은 주 링크 없음

## 2. 가격과 이어진 링크

시차는 가격 시점 기준(−2 = 그 변수의 2주 전 값, +1 = 1주 뒤 값). 강도는 편상관.

| 분류 | 끝점 표시 (변수 쪽, 가격 쪽) |
| --- | --- |
| 원인 | `변수 --> 가격` |
| 원인 후보 | `변수 o-> 가격` (LPCMCI) |
| 결과 | `가격 --> 변수` |
| 결과 후보 | `가격 o-> 변수` (LPCMCI) |
| 잠재 교란 | `변수 <-> 가격` — 관측되지 않은 공통 원인 (LPCMCI) |
| 방향 미정 | 그 밖 (`o-o` 등) |

`답지 인용` 은 답지 전체 기간에 그 변수가 원인으로 인용된 주의 수.

In [ ]:
HEAD = set("<>")


def price_links(g, val):
    """가격(0번)과 이어진 링크: (변수 번호, 시차, 변수 쪽 표시, 가격 쪽 표시, 강도)."""
    out = []
    for i, tau in zip(*np.nonzero(g[:, 0, :] != "")):  # X^i(t-tau) — 가격(t). tau=0 도 여기서 한 번만 센다
        out.append((int(i), -int(tau), g[i, 0, tau][0], g[i, 0, tau][2], val[i, 0, tau]))
    for j, tau in zip(*np.nonzero(g[0, :, 1:] != "")):  # 가격(t) — X^j(t+tau)
        out.append((int(j), int(tau) + 1, g[0, j, tau + 1][2], g[0, j, tau + 1][0], val[0, j, tau + 1]))
    return [x for x in out if x[0] != 0]  # 가격 자기 시차는 뺀다


def kind(at_var, at_price):
    if at_price == ">" and at_var == "-":
        return "원인"
    if at_price == ">" and at_var == "o":
        return "원인 후보"
    if at_var in HEAD and at_price == "-":
        return "결과"
    if at_var in HEAD and at_price == "o":
        return "결과 후보"
    if at_var in HEAD and at_price in HEAD:
        return "잠재 교란"
    return "방향 미정"


def spouses(g, links):
    """가격이 화살촉으로 들어가는 이웃 j 의, 가격 말고 j 에 화살촉으로 들어오는 다른 변수들."""
    out = set()
    for j, _, at_var, _, _ in links:
        if at_var in HEAD:
            heads = {int(k) for k, tau in zip(*np.nonzero(g[:, j, :] != "")) if g[k, j, tau][2] == ">"}
            out |= heads - {0, j}
    return out


def blanket_sets(g, val, cols):
    links = price_links(g, val)
    cause = {cols[i] for i, _, a, b, _ in links if kind(a, b) in ("원인", "원인 후보")}
    mb = {cols[i] for i, *_ in links} | {cols[k] for k in spouses(g, links)}
    return cause, mb


def link_table(s, m):
    g, val, _ = results[s, m]
    cols = list(panels[s].columns)
    cites = keys[s].cited.explode().value_counts()
    arrow = lambda a, b: ("<" if a in HEAD else a) + "-" + (">" if b in HEAD else b)  # 변수 쪽을 왼쪽에 쓴다
    rows = [(cols[i][:60], o, f"변수 {arrow(a, b)} 가격", kind(a, b), v, int(cites.get(cols[i], 0)))
            for i, o, a, b, v in sorted(price_links(g, val), key=lambda x: (x[1], x[0]))]
    return pd.DataFrame(rows, columns=["변수", "시차(주)", "링크", "분류", "강도", "답지 인용"]).round(3)


for s in SERIES:
    for m in METHODS:
        print(f"\n===== {SERIES[s]}  {m}")
        print(link_table(s, m).to_string(index=False))

## 3. 답지 채점

집합 두 가지로 센다.
- **원인 집합**: 원인 + 원인 후보의 변수.
- **MB 집합**: 가격과 이어진 모든 변수 + 가격이 화살촉으로 들어가는 이웃의 다른 화살촉 부모(배우자). PCMCI 에서는 보통의 Markov blanket 이고, RCBNB-MB 노트북의 적중률과 같은 정의다.

W = 답지 기간 중 인용 변수가 분석에 하나 이상 들어 있는 주, C_w = 그 주의 인용 변수, S = 집합, N = 가격 외 변수 수.

- 원인 적중률 = (1/|W|) Σ_w 1[C_w ∩ S ≠ ∅]
- 우연 기준선 = 같은 크기 |S| 의 무작위 변수 집합의 기대 적중률 = (1/|W|) Σ_w [1 − C(N − |C_w|, |S|) / C(N, |S|)] (초기하분포)
- precision / recall / F1 = 답지 기간 인용 변수 전체 vs S

In [ ]:
def chance_hit(n_cited, m, N):
    return 1 - comb(N - n_cited, m) / comb(N, m)


assert chance_hit(3, 10, 10) == 1 and chance_hit(3, 0, 10) == 0 and abs(chance_hit(1, 1, 4) - 0.25) < 1e-12


def score(S, s):
    X = panels[s]
    cols, N = set(X.columns) - {"price"}, X.shape[1] - 1
    cited = keys[s].cited[keys[s].index.isin(X.index)].map(lambda v: v & cols)
    cited = cited[cited.map(bool)]
    truth = set().union(*cited)
    tp, fp, fn = len(truth & S), len(S - truth), len(truth - S)
    return {"집합 크기": len(S), "답지 주 수": len(cited),
            "원인 적중률": np.mean([bool(c & S) for c in cited]),
            "우연 기준선": np.mean([chance_hit(len(c), len(S), N) for c in cited]),
            "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1), "F1": 2 * tp / max(2 * tp + fp + fn, 1)}


sets = {(s, m): blanket_sets(*results[s, m][:2], list(panels[s].columns)) for s, m in jobs}
summary = pd.DataFrame([{"원자재": s, "방법": m, "집합": name, **score(S, s)}
                        for (s, m), pair in sets.items() for name, S in zip(["원인", "MB"], pair)])
summary["적중률 − 우연"] = summary["원인 적중률"] - summary["우연 기준선"]
summary.round(3)

참고 — `RCBNB-MB.ipynb` 6절의 레짐 없는 CBNB(PCMCI+ + LiNGAM, MB 집합) 결과: 커피 0.593, 구리 0.398, 금 0.356.
(2026-11-08 스냅샷. 그쪽 노트북을 다시 돌리면 거기 표가 기준이다.)

## 4. 그림 — 적중률과 우연 기준선

점 사이 거리가 "같은 크기의 무작위 집합보다 얼마나 더 맞혔는가"다. 집합이 클수록 우연 기준선도 올라간다.

In [ ]:
have = {f.name for f in font_manager.fontManager.ttflist}  # 한글 글꼴: 설치된 첫 번째 것
plt.rcParams.update({"font.family": next((f for f in ("AppleGothic", "Malgun Gothic", "NanumGothic") if f in have), "sans-serif"),
                     "axes.unicode_minus": False})
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
HIT, CHANCE = "#2a78d6", "#eb6834"

fig, axes = plt.subplots(1, len(SERIES), figsize=(13, 3.6), sharex=True, facecolor=SURFACE)
for ax, s in zip(axes, SERIES):
    d = summary[summary.원자재 == s].reset_index(drop=True)
    y = np.arange(len(d))[::-1]
    ax.hlines(y, d["우연 기준선"], d["원인 적중률"], color=GRID, lw=2, zorder=1)
    ax.scatter(d["우연 기준선"], y, s=70, color=CHANCE, edgecolor=SURFACE, linewidth=2, zorder=2, label="우연 기준선")
    ax.scatter(d["원인 적중률"], y, s=70, color=HIT, edgecolor=SURFACE, linewidth=2, zorder=3, label="원인 적중률")
    for yy, h in zip(y, d["원인 적중률"]):
        ax.annotate(f"{h:.2f}", (h, yy), xytext=(0, 9), textcoords="offset points", ha="center", color=INK, fontsize=9)
    ax.set_yticks(y, [f"{m} {n} ({k}개)" for m, n, k in zip(d.방법, d.집합, d["집합 크기"])])
    ax.set_title(SERIES[s], loc="left", color=INK)
    ax.set_xlim(0, 1.05)
    ax.set_ylim(-0.6, len(d) - 0.2)
    ax.set_facecolor(SURFACE)
    ax.grid(axis="x", color=GRID, lw=0.8)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.tick_params(colors=MUTED, length=0)
axes[0].legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.08), ncol=2, labelcolor=MUTED)
fig.suptitle("답지 원인 적중률 vs 같은 크기 무작위 집합", x=0.01, ha="left", color=INK, y=1.12)
plt.show()